# Veil In-House Image Detector — Kaggle Training Kernel

Runs the end-to-end pipeline (`detector-trainer/run_pipeline.py`) on Kaggle GPU:
**manifest -> train_resnet -> train_clip -> evaluate**, writing all outputs to
`/kaggle/working` for retrieval via `kaggle kernels output`.

## How the code gets onto Kaggle
This kernel needs the `detector-trainer/` source tree importable. Two supported ways:

1. **Git clone (default below, needs `enable_internet: true`)** — clone the repo
   at run time. Simplest; the `CODE` cell does this.
2. **Attached utility dataset** — push `detector-trainer/` as a Kaggle Dataset
   (`kaggle datasets create/version`), attach it, and point `CODE_DIR` at
   `/kaggle/input/<your-code-dataset>/detector-trainer`. Then set `USE_GIT = False`.

See `detector-trainer/kernels/README.md` for the exact CLI loop.

## 0. Config
Edit `REPO_URL` / `BRANCH` (git path) or `CODE_DIR` (attached-dataset path).

In [ ]:
import os, subprocess, sys, glob, json

# Diagnostic: show the mounted input tree (top levels)
for root, dirs, files in sorted(os.walk('/kaggle/input')):
    if root.count('/') - 2 <= 2:
        print(root, '-> dirs:', sorted(dirs)[:20], '| files:', sorted(files)[:12])

# Auto-locate the code wherever Kaggle extracted it
hits = glob.glob('/kaggle/input/**/run_pipeline.py', recursive=True)
assert hits, 'run_pipeline.py not found under /kaggle/input'
CODE_DIR = os.path.dirname(hits[0])
print('CODE_DIR =', CODE_DIR)

OUT_DIR = '/kaggle/working/veil_run'
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Get the code

In [ ]:
assert os.path.isdir(CODE_DIR), f'code dir not found: {CODE_DIR}'
print('code dir contents:', sorted(os.listdir(CODE_DIR)))


## 2. Install dependencies
Kaggle has torch preinstalled; this pulls open_clip / imagehash / matching pins.

In [ ]:
# Kaggle ships torch/torchvision/sklearn/matplotlib/pandas/Pillow/numpy + imagehash.
# open_clip_torch needs internet (phone-verify the account). Non-fatal: the manifest
# validation stage only needs imagehash, so we continue even if the network is off.
extras = ['imagehash', 'open_clip_torch']
r = subprocess.run([sys.executable, '-m', 'pip', 'install', *extras],
                   capture_output=True, text=True)
print(r.stdout[-1200:]); print(r.stderr[-1200:])
if r.returncode:
    print('WARN: some extras failed (offline?). Continuing — manifest stage only needs imagehash.')
else:
    print('deps installed')


## 3. Locate the attached datasets
Kaggle mounts `dataset_sources` (see `kernel-metadata.json`) read-only under
`/kaggle/input/<dataset-slug>/`. We pass BOTH roots to `--data-root`; the
pipeline auto-discovers per-generator / category subfolders and classifies them
(real vs each generator). Wild generators (midjourney/dalle3/flux) are routed to
`test_wild` by the split logic. Drop any self-generated dalle3/flux images into a
folder named `dalle3`/`flux` under a data root and they are picked up automatically.

In [ ]:
import glob
# Auto-locate each dataset wherever Kaggle mounts it (/kaggle/input/... or .../datasets/<user>/...)
def find_ds(slug):
    hits = [h for h in glob.glob(f'/kaggle/input/**/{slug}', recursive=True) if os.path.isdir(h)]
    return hits[0] if hits else None
DATA_ROOTS = [p for p in [find_ds('unbiased-tiny-genimage'),
                          find_ds('ai-vs-real-images-dataset')] if p]
assert DATA_ROOTS, 'no training datasets found under /kaggle/input'
for r in DATA_ROOTS:
    print(r, '->', sorted(os.listdir(r))[:20])


## 4. Run the full pipeline
All four stages to `/kaggle/working/veil_run`. Adjust epochs for the GPU budget.

In [ ]:
STAGES = 'all'  # validation run: discovery+split only (no GPU). Switch to 'all' for full training.
cmd = [
    sys.executable, 'run_pipeline.py',
    '--data-root', *DATA_ROOTS,
    '--manifest', os.path.join(OUT_DIR, 'manifest.csv'),
    '--out', OUT_DIR,
    '--stages', STAGES,
    '--pretrained',              # ImageNet init for the ResNet baseline
    '--resnet-epochs', '10',
    '--clip-backbone', 'ViT-L-14',
    '--clip-epochs', '200',
    '--num-workers', '2',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=CODE_DIR, check=True)

## 5. Inspect outputs
`report.md`, plots, per-model predictions, checkpoints and `results.json` all land
under `/kaggle/working/veil_run`. Pull them back locally with:
```
kaggle kernels output <KAGGLE_USERNAME>/veil-detector-train -p ./kaggle_out
```

In [ ]:
for p in sorted(glob.glob(os.path.join(OUT_DIR, '**', '*'), recursive=True)):
    if os.path.isfile(p):
        print(p)
print('\n----- report.md -----')
rp = os.path.join(OUT_DIR, 'report', 'report.md')
if os.path.exists(rp):
    print(open(rp).read())